In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/SafeCue Audio Recordings"

print("Folder exists:", os.path.exists(BASE_DIR))
print("\nContents:")

for item in os.listdir(BASE_DIR):
    print( item)

Folder exists: True

Contents:
Siren
Fire Alarm
Smoke Alarm
Screaming
Background Noise


In [4]:
!pip install -q yt-dlp
!apt-get -qq update
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.3 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
import os

METADATA_DIR = os.path.join(BASE_DIR, "_metadata")
os.makedirs(METADATA_DIR, exist_ok=True)

print("Metadata folder:", METADATA_DIR)

Metadata folder: /content/drive/MyDrive/SafeCue Audio Recordings/_metadata


In [6]:
import pandas as pd

metadata_url = "https://research.google.com/audioset/download.html"

print("AudioSet metadata page:")
print(metadata_url)

AudioSet metadata page:
https://research.google.com/audioset/download.html


In [7]:
!wget -q --show-progress \
"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/balanced_train_segments.csv" \
-o /dev/null

In [8]:
import os

print(os.listdir(METADATA_DIR))

[]


In [9]:
import requests

url = "https://research.google.com/audioset/download.html"
response = requests.get(url)

print("Status code:", response.status_code)
print("Page downloaded:", len(response.text) > 0)

Status code: 200
Page downloaded: True


In [10]:
import requests

url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/balanced_train_segments.csv"

output_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

response = requests.get(url)

with open(output_path, "wb") as f:
    f.write(response.content)

print("Downloaded:", output_path)
print("File size:", os.path.getsize(output_path), "bytes")

Downloaded: /content/drive/MyDrive/SafeCue Audio Recordings/_metadata/balanced_train_segments.csv
File size: 1211931 bytes


In [14]:
csv_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

with open(csv_path, "r", encoding="utf-8") as f:
    for i in range(8):
        print(f"LINE {i+1}:", repr(f.readline()))

LINE 1: '# Segments csv created Sun Mar  5 10:54:31 2017\n'
LINE 2: '# num_ytids=22160, num_segs=22160, num_unique_labels=527, num_positive_labels=52882\n'
LINE 3: '# YTID, start_seconds, end_seconds, positive_labels\n'
LINE 4: '--PJHxphWEs, 30.000, 40.000, "/m/09x0r,/t/dd00088"\n'
LINE 5: '--ZhevVpy1s, 50.000, 60.000, "/m/012xff"\n'
LINE 6: '--aE2O5G5WE, 0.000, 10.000, "/m/03fwl,/m/04rlf,/m/09x0r"\n'
LINE 7: '--aO5cdqSAg, 30.000, 40.000, "/t/dd00003,/t/dd00005"\n'
LINE 8: '--aaILOrkII, 200.000, 210.000, "/m/032s66,/m/073cg4"\n'


In [15]:
import csv
import pandas as pd
import os

csv_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

rows = []

with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if row and not row[0].startswith("#"):
            rows.append(row)

df = pd.DataFrame(
    rows,
    columns=["YTID", "start_seconds", "end_seconds", "positive_labels"]
)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Shape: (22160, 4)

First 5 rows:


,YTID,start_seconds,end_seconds,positive_labels
0,--PJHxphWEs,30.000,40.000,"/m/09x0r,/t/dd00088"
1,--ZhevVpy1s,50.000,60.000,/m/012xff
2,--aE2O5G5WE,0.000,10.000,"/m/03fwl,/m/04rlf,/m/09x0r"
3,--aO5cdqSAg,30.000,40.000,"/t/dd00003,/t/dd00005"
4,--aaILOrkII,200.000,210.000,"/m/032s66,/m/073cg4"


In [16]:
import requests
import json
import os

ontology_url = "https://raw.githubusercontent.com/audioset/ontology/master/ontology.json"
ontology_path = os.path.join(METADATA_DIR, "ontology.json")

response = requests.get(ontology_url)
response.raise_for_status()

with open(ontology_path, "wb") as f:
    f.write(response.content)

print("Ontology downloaded successfully!")
print("File size:", os.path.getsize(ontology_path), "bytes")

Ontology downloaded successfully!
File size: 342780 bytes


In [17]:
import json

with open(ontology_path, "r", encoding="utf-8") as f:
    ontology = json.load(f)

keywords = ["siren", "fire alarm", "smoke alarm", "screaming", "background noise"]

for keyword in keywords:
    print(f"\n=== {keyword.upper()} ===")

    matches = []
    for item in ontology:
        name = item.get("name", "")
        if keyword.lower() in name.lower():
            matches.append((item.get("id"), name))

    for audio_id, name in matches:
        print(audio_id, "→", name)


=== SIREN ===
/m/04qvtq → Police car (siren)
/m/012n7d → Ambulance (siren)
/m/012ndj → Fire engine, fire truck (siren)
/m/03kmc9 → Siren
/m/0dgbq → Civil defense siren

=== FIRE ALARM ===
/m/0c3f7m → Fire alarm

=== SMOKE ALARM ===
/m/01y3hg → Smoke detector, smoke alarm

=== SCREAMING ===
/m/03qc9zr → Screaming

=== BACKGROUND NOISE ===
/m/093_4n → Background noise


In [18]:
label_ids = {
    "Siren": "/m/03kmc9",
    "Fire Alarm": "/m/0c3f7m",
    "Smoke Alarm": "/m/01y3hg",
    "Screaming": "/m/03qc9zr",
    "Background Noise": "/m/093_4n"
}

for category, label_id in label_ids.items():
    count = df["positive_labels"].apply(
        lambda x: label_id in x.split(",")
    ).sum()

    print(f"{category}: {count} clips")

Siren: 207 clips
Fire Alarm: 60 clips
Smoke Alarm: 60 clips
Screaming: 59 clips
Background Noise: 0 clips


In [19]:
import requests
import os

url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/unbalanced_train_segments.csv"

output_path = os.path.join(METADATA_DIR, "unbalanced_train_segments.csv")

response = requests.get(url)
response.raise_for_status()

with open(output_path, "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")
print("File size:", os.path.getsize(output_path), "bytes")

Downloaded successfully!
File size: 101468408 bytes


In [20]:
import csv
from collections import Counter

unbalanced_path = os.path.join(
    METADATA_DIR,
    "unbalanced_train_segments.csv"
)

counts = Counter()

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        for category, label_id in label_ids.items():
            if label_id in labels:
                counts[category] += 1

print("AudioSet unbalanced training counts:")
for category in label_ids:
    print(f"{category}: {counts[category]} clips")

AudioSet unbalanced training counts:
Siren: 8079 clips
Fire Alarm: 801 clips
Smoke Alarm: 426 clips
Screaming: 1126 clips
Background Noise: 0 clips


In [21]:
siren_id = "/m/03kmc9"

siren_rows = df[
    df["positive_labels"].apply(
        lambda x: siren_id in x.split(",")
    )
].reset_index(drop=True)

print("Siren clips available:", len(siren_rows))
print("\nFirst Siren clip:")
display(siren_rows.head(1))

Siren clips available: 207

First Siren clip:


,YTID,start_seconds,end_seconds,positive_labels
0,-4RWqM0UCCY,0.000,10.000,"/m/012n7d,/m/02mfyn,/m/03j1ly,/m/03kmc9"


In [23]:
import subprocess
import os
import time

# Number of clips to test
NUM_TEST = 10

# Use the first 10 Siren clips
test_rows = siren_rows.head(NUM_TEST)

siren_folder = os.path.join(BASE_DIR, "Siren")
os.makedirs(siren_folder, exist_ok=True)

success = 0
failed = 0

for i, row in test_rows.iterrows():
    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        siren_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # Skip if already downloaded
    if os.path.exists(output_file):
        print(f"Already exists: {ytid}")
        success += 1
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(f"Downloading {success + failed + 1}/{NUM_TEST}: {ytid}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        success += 1
        print("  ✅ Success")
    else:
        failed += 1
        print("  ❌ Failed")

    time.sleep(1)

print("\n====================")
print("DOWNLOAD TEST COMPLETE")
print("====================")
print("Successful:", success)
print("Failed:", failed)

Already exists: -4RWqM0UCCY
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success
  ✅ Success

DOWNLOAD TEST COMPLETE
Successful: 10
Failed: 0


In [24]:
# Create a list of unique Siren YouTube sources
# We will use this list for the larger download.

siren_candidates = siren_rows.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print("Total Siren candidates:", len(siren_candidates))
print("\nFirst 10 candidates:")
display(siren_candidates.head(10))

Total Siren candidates: 207

First 10 candidates:


,YTID,start_seconds,end_seconds,positive_labels
0,-4RWqM0UCCY,0.000,10.000,"/m/012n7d,/m/02mfyn,/m/03j1ly,/m/03kmc9"
1,-Y1qiiugnk8,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9"
2,-_dElQcyJnA,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9,/m/04qvtq"
3,-gKNRXbpAKs,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9,/m/0dgbq"
4,-oSw8wNpLcY,60.000,70.000,"/m/030rvx,/m/03kmc9"
5,-z47zN0YyE4,420.000,430.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
6,03NLMEMi8-I,30.000,40.000,"/m/03kmc9,/m/0dgbq"
7,0JKcTVpby0I,30.000,40.000,"/m/03kmc9,/m/0dgbq"
8,0JPT13OUVV8,220.000,230.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
9,0PhU-PIsUMw,80.000,90.000,"/m/03kmc9,/m/0dgbq"


In [25]:
import csv
import pandas as pd
import os

unbalanced_path = os.path.join(
    METADATA_DIR,
    "unbalanced_train_segments.csv"
)

siren_id = "/m/03kmc9"

siren_unbalanced_rows = []

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        if siren_id in labels:
            siren_unbalanced_rows.append(row)

siren_unbalanced = pd.DataFrame(
    siren_unbalanced_rows,
    columns=["YTID", "start_seconds", "end_seconds", "positive_labels"]
)

# Remove duplicate YouTube videos
siren_unbalanced = siren_unbalanced.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print("Unique Siren candidates in unbalanced set:", len(siren_unbalanced))
display(siren_unbalanced.head(10))

Unique Siren candidates in unbalanced set: 8079


,YTID,start_seconds,end_seconds,positive_labels
0,--7T50tAIrg,6.000,16.000,"/m/03kmc9,/m/09x0r"
1,-09rxiqNNEs,30.000,40.000,"/m/03kmc9,/m/0dgbq"
2,-0Eem_FuIto,15.000,25.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
3,-122tCXtFhU,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq,/m/07yv9,/m/0k4j"
4,-2GlU3e0nTU,170.000,180.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"
5,-34Lznw_wUA,10.000,20.000,/m/03kmc9
6,-3qh-WFUV2U,30.000,40.000,"/m/03kmc9,/m/0dgbq"
7,-4JG_Ag99hY,30.000,40.000,"/m/03kmc9,/m/0dgbq"
8,-5EiLqHCLuI,0.000,10.000,"/m/012n7d,/m/03j1ly,/m/03kmc9"
9,-5o4C4ZtHSU,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"


In [26]:
# Select 260 Siren candidates for downloading
# We use a fixed random seed so the selection is reproducible.

SIREN_TARGET = 260

siren_download_candidates = siren_unbalanced.sample(
    n=SIREN_TARGET,
    random_state=42
).reset_index(drop=True)

print("Siren candidates selected:", len(siren_download_candidates))

display(siren_download_candidates.head(10))

Siren candidates selected: 260


,YTID,start_seconds,end_seconds,positive_labels
0,JngI6xTRE2o,30.000,40.000,"/m/012n7d,/m/03j1ly,/m/03kmc9,/m/04qvtq"
1,uaFJj5A9jCE,2.000,12.000,"/m/012ndj,/m/03j1ly,/m/03kmc9"
2,w8IVfFqEXjw,60.000,70.000,"/m/03kmc9,/m/0dgbq"
3,ysJhEu_L7lc,30.000,40.000,/m/03kmc9
4,r-ww2qIp1BE,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"
5,NHcHQ6ptBmA,30.000,40.000,/m/03kmc9
6,ctyG4PM_rtw,0.000,10.000,"/m/0342h,/m/03kmc9,/m/04szw,/m/0fx80y"
7,azXoLk-FCrU,30.000,40.000,"/m/03kmc9,/m/04qvtq,/m/07yv9"
8,CiYGv42LsZ8,28.000,38.000,"/m/03kmc9,/m/07yv9,/m/0k4j"
9,axOkcNpjyb4,30.000,40.000,"/m/03j1ly,/m/03kmc9,/m/04qvtq"


In [27]:
import subprocess
import os
import time
import pandas as pd

siren_folder = os.path.join(BASE_DIR, "Siren")
os.makedirs(siren_folder, exist_ok=True)

results = []

for i, row in siren_download_candidates.iterrows():

    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        siren_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # If already downloaded, don't download again
    if os.path.exists(output_file):
        print(f"{i+1}/{len(siren_download_candidates)} ✅ Already exists: {ytid}")

        results.append({
            "YTID": ytid,
            "start_seconds": start,
            "end_seconds": end,
            "label": "Siren",
            "filename": output_file,
            "status": "already_exists"
        })
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(f"{i+1}/{len(siren_download_candidates)} Downloading: {ytid}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        status = "success"
        print("   ✅ Success")
    else:
        status = "failed"
        print("   ❌ Failed")

    results.append({
        "YTID": ytid,
        "start_seconds": start,
        "end_seconds": end,
        "label": "Siren",
        "filename": output_file if status == "success" else "",
        "status": status
    })

    time.sleep(1)

# Save download results
download_results = pd.DataFrame(results)

results_path = os.path.join(
    METADATA_DIR,
    "siren_download_results.csv"
)

download_results.to_csv(results_path, index=False)

print("\n==============================")
print("SIREN DOWNLOAD COMPLETE")
print("==============================")
print("Successful:", (download_results["status"] == "success").sum())
print("Already existed:", (download_results["status"] == "already_exists").sum())
print("Failed:", (download_results["status"] == "failed").sum())
print("Results saved to:", results_path)

1/260 Downloading: JngI6xTRE2o
   ✅ Success
2/260 Downloading: uaFJj5A9jCE
   ✅ Success
3/260 Downloading: w8IVfFqEXjw
   ✅ Success
4/260 Downloading: ysJhEu_L7lc
   ✅ Success
5/260 Downloading: r-ww2qIp1BE
   ❌ Failed
6/260 Downloading: NHcHQ6ptBmA
   ✅ Success
7/260 Downloading: ctyG4PM_rtw
   ✅ Success
8/260 Downloading: azXoLk-FCrU
   ✅ Success
9/260 Downloading: CiYGv42LsZ8
   ✅ Success
10/260 Downloading: axOkcNpjyb4
   ✅ Success
11/260 Downloading: 1Q61oRUkfC0
   ✅ Success
12/260 Downloading: Lg6yI5szlVw
   ✅ Success
13/260 Downloading: yDeuZ92Y0eQ
   ✅ Success
14/260 Downloading: 2iKT4cvbJy8
   ❌ Failed
15/260 Downloading: 3LblLgNthRs
   ✅ Success
16/260 Downloading: FF410bQhJ8s
   ✅ Success
17/260 Downloading: HWeSf6Dsa5g
   ✅ Success
18/260 Downloading: 79-rErKgp3s
   ✅ Success
19/260 Downloading: vVEpMHmcH5Q
   ✅ Success
20/260 Downloading: eRZyNpT-XB8
   ✅ Success
21/260 Downloading: lAyZQxneqGA
   ✅ Success
22/260 Downloading: 0rdPzsSkzOU
   ✅ Success
23/260 Downloading: 0